# Kolokvijum II (Septembar 1) — Workflow i Task

Ovaj notebook objašnjava rešenje iz **`k2_provere_na_kraju.html`**: koje koncepte funkcionalnog
programiranja zadatak traži, kojim mehanizmima su ostvareni i po čemu se to razlikuje od objektnog pristupa.

## Tekst zadatka

> Napraviti konstruktore za objekte tipa **Workflow** i **Task**. Svaki Task opisan je nazivom, definicijom i metodom
> **execute**. Definicija predstavlja funkciju koja se izvršava pozivom funkcije execute. Rezultat poziva metode
> execute nad Task objektom je rezultat poziva funkcije date u atributu definicija. Metoda execute može primiti
> **proizvoljan broj argumenata** koji prosleđuje funkciji definisanoj u atributu definicija.
> Workflow sadrži podatke naziv, autor, spisak Task objekata i metodu execute. Workflow je **iterabilni objekat**,
> iterisanjem kroz workflow dobijaju se pojedinačni Task objekti. Za Workflow definisati operacije
> **map, flat, flatMap i filter**.
>
> Metoda execute izvršava sve zadate Task objekte tako što kao ulaz u execute metodu Task objekta dostavlja
> **rezultat prethodno izvršenog** Task objekta. Za ulaz u metodu za prvi task objekat uzimaju se **svi prosleđeni
> argumenti** iz metode execute u Workflow objektu. Povratna vrednost metode je rezultat izvršavanja **poslednjeg**
> Task objekta iz niza. Execute metode iz Workflow i Task objekta se izvršavaju **asinhrono**.
>
> Modifikovati Workflow objekat tako da je nad njim moguće primeniti **proizvoljnu kompoziciju transdjusera**.
> Demonstrirati ispravnost implementacije definisanjem transdjusera za **ispis svakog koraka** izvršavanja Workflow objekta.
>
> Demonstrirati ispravnost rešenja instanciranjem nekoliko Workflow i Task objekata i pozivanjem metoda definisanih nad njima.

## Podela na logičke jedinice

| Jedinica | Šta sadrži | Glavni mehanizam | Koncept FP-a |
| --- | --- | --- | --- |
| 1 | `Task` | fabrička funkcija, `async`, rest parametri | funkcija kao podatak |
| 2 | transdjuseri | `map`, `filter`, `compose`, krajnji redjuseri | kompozicija, odvajanje „šta" od „kako" |
| 3 | `izravnaj`, `izvrsiRedom` | iterabilni protokol, `reduce` nad `Promise` | asinhrono preklapanje |
| 4 | `Workflow` | generator `Symbol.iterator`, `Object.freeze` | iterabilnost, nepromenljivost |
| 5 | `saIspisom`, `merenje` | transdjuser koji obmotava zadatak | dekorisanje bez izmene originala |
| P | provere | poređenje dobijenog i očekivanog | testiranje asinhronog koda |

## Raspored rešenja

- **DEO I — IMPLEMENTACIJA** · samo definicije, nijedan ispis.
- **DEO II — PROVERE** · instanciranje, pozivi i sav ispis, na kraju, sa izveštajem `N/N provera prošlo`.

**Napomena o `await`.** U običnom `<script>` nema `await` na najvišem nivou, pa su u HTML fajlu sve provere
spakovane u asinhronu funkciju `provere()` koja se poziva na dnu. U ovom notebook-u Deno kernel dozvoljava
`await` na najvišem nivou, pa su ćelije pisane direktno — isti kod, bez omotača.

Odluke koje tekst zadatka ne propisuje označene su sa **Pretpostavka**.

## Pravila funkcionalne paradigme kojih se rešenje drži

| Pravilo | Kako je sprovedeno u ovom rešenju |
| --- | --- |
| **funkcije su podaci** | `definicija` je obično polje objekta; transdjuseri su funkcije koje se prosleđuju i slažu |
| **nepromenljivost** | `Object.freeze` nad `Task`-om, nad `Workflow`-om i nad spiskom zadataka; `map`/`filter`/`flat`/`flatMap` vraćaju **nov** Workflow |
| **bez `this` i bez `new`** | fabričke funkcije; jedini `this` bio bi u generatoru, pa je i on pisan tako da mu ne treba |
| **čiste funkcije** | `izravnaj`, `compose`, `map`, `filter`, sve četiri operacije nad Workflow-om |
| **kompozicija umesto nasleđivanja** | ponašanje se dodaje slaganjem transdjusera, ne izvođenjem podklase |
| **preklapanje umesto petlje** | `reduce` i za spisak zadataka i za lanac `Promise`-a; nigde `for` osim `for…of` u proveri |
| **odvojeno „šta" od „kako"** | transdjuser opisuje transformaciju; krajnji redjuser odlučuje u šta se skuplja |
| **efekti na ivici** | `console.log` samo u DEO II i unutar transdjusera koji je za ispis i napravljen |

### Svesni izuzetak

`saIspisom` i `merenje` **jesu** nečisti — postoje da bi ispisivali. To je i smisao: prljav posao je izdvojen u
zaseban transdjuser koji se dodaje ili sklanja jednim potezom, umesto da bude razasut po `execute` metodama.
Original ostaje čist; obmotana verzija je poseban objekat.

---
# DEO I — IMPLEMENTACIJA

Sve ćelije ispod su **samo definicije**. Instanciranje i ispis dolaze tek u DEO II.

---
# Jedinica 1 · `Task` — naziv, definicija i asinhrona metoda `execute`

In [ ]:
// Definicija je obična funkcija koja se čuva kao podatak.
// execute prima PROIZVOLJAN broj argumenata i prosleđuje ih definiciji;
// rezultat je rezultat poziva definicije. Pošto je execute async,
// uvek vraća Promise — i kada je definicija sinhrona.
const Task = (naziv, definicija) => Object.freeze({
    naziv,
    definicija,

    execute: async (...argumenti) => definicija(...argumenti)
});

### Koncepti

**Funkcija kao podatak (prvorazredna funkcija).** `definicija` stoji u objektu kao i `naziv` — može se pročitati,
proslediti, obmotati ili zameniti. Ceo zadatak stoji na toj jednoj osobini: bez nje `Task` bi morao da bude
apstraktna klasa sa metodom koju svaki zadatak nasleđuje.

**Ponašanje kroz vrednost, ne kroz nasleđivanje.** Nov zadatak je nov **poziv fabrike**, ne nova klasa.
`Task("zbir", niz => niz.reduce(...))` je ceo „podtip".

**Asinhronost kao jednoobrazan omotač.** `async` čini da `execute` **uvek** vrati `Promise`, bez obzira na to da li
je definicija sinhrona. Zahvaljujući tome pozivalac nikada ne mora da pita „da li je ovaj zadatak spor" —
svi se tretiraju isto, pa se sinhroni i asinhroni koraci mogu slobodno mešati u istom lancu.

### Mehanizmi

**Rest parametri `(...argumenti)`** — skupljaju proizvoljan broj argumenata u niz.

**Spread pri prosleđivanju `definicija(...argumenti)`** — razlaže ih nazad u pojedinačne argumente.
Par rest/spread je ovde tačan prevod zahteva „proizvoljan broj argumenata koje prosleđuje definiciji".

**`async` strelica** — implicitno pakuje povratnu vrednost u `Promise.resolve(...)`, a ako je definicija sama
asinhrona, `Promise` se ne ugnježđuje nego se ravna sam od sebe.

**`Object.freeze`** — zadatak posle nastanka nema svoj tok; `saIspisom` ga ne menja nego pravi **nov** Task.

### Pretpostavka

Tekst ne kaže šta se dešava ako definicija baci grešku. Pošto je `execute` `async`, greška izlazi kao
**odbijen `Promise`**, pa `izvrsiRedom` prekida lanac i greška stiže do pozivaoca `execute`-a nad Workflow-om.
Tako se ponaša i `demo`/`provere` omotač sa `.catch`.

### Funkcionalna paradigma prema OOP

| | Ovde (FP) | Kako bi bilo u OOP |
| --- | --- | --- |
| definisanje posla | funkcija u polju | apstraktna metoda koju podklasa implementira |
| nov zadatak | poziv fabrike | nova klasa + `new` |
| proizvoljni argumenti | rest/spread | preopterećenje metode ili `Object[] args` |
| asinhronost | `async` na jednom mestu | `Future`/`Task` tip kroz celu hijerarhiju |

**Prednosti.** Nema hijerarhije klasa: deset zadataka je deset redova. Zadatak se lako pravi u toku rada,
od podataka. Nema `this`, pa se `execute` sme proslediti kao vrednost.

**Mane.** Nema `instanceof Task` ni provere da definicija zaista prima ono što joj se šalje — greška izlazi
tek pri izvršavanju. Naziv je samo string, ne i tip.

---
# Jedinica 2 · Transdjuseri — alat za proizvoljnu kompoziciju

In [ ]:
const collect = (acc, e) => [...acc, e];                            // krajnji redjuser — rezultat je niz
const sum     = (acc, e) => acc + e;                                // krajnji redjuser — zbir
const map     = f => next => (acc, e) => next(acc, f(e));           // transdjuser — preslikavanje
const filter  = p => next => (acc, e) => p(e) ? next(acc, e) : acc; // transdjuser — izdvajanje

// compose(a, b)(krajnji) = a(b(krajnji)) — podatak teče kroz a, pa kroz b
const compose = (...fs) => x => fs.reduceRight((acc, f) => f(acc), x);

### Koncepti

**Redjuser.** Funkcija oblika `(akumulator, element) => noviAkumulator`. To je jedini oblik koji `reduce` razume,
i dovoljan je da opiše i skupljanje u niz (`collect`) i sabiranje (`sum`) i bilo šta drugo.

**Transdjuser.** Funkcija koja **prima redjuser i vraća redjuser**: `next => (acc, e) => …`.
Ne zna gde se skuplja niti šta je akumulator — samo šta radi sa jednim elementom pre nego što ga preda dalje.
Zbog toga je `map(f)` upotrebljiv i kad se skuplja u niz i kad se sabira, bez ijedne izmene.

**Odvajanje „šta" od „kako".** `map(f)` kaže **šta** se radi sa elementom; `collect`/`sum` kažu **kako** se
rezultat skuplja; `reduce` odlučuje **odakle** elementi dolaze. Tri odluke koje u običnom `niz.map(f).filter(p)`
stoje zapetljane, ovde su razdvojene.

**Kompozicija.** Dva transdjusera se spajaju u jedan, i spoj je opet transdjuser. Zbog toga je „proizvoljna
kompozicija" iz teksta zadatka doslovno moguća — nema ograničenja koliko ih se sme nanizati.

**Jedan prolaz, bez međunizova.** `niz.filter(p).map(f)` pravi međuniz i prolazi dva puta.
`reduce(compose(filter(p), map(f))(collect), [])` prolazi **jednom** i međuniz ne postoji.

### Mehanizmi

**Ulančane strelice `f => next => (acc, e) => …`** — tri nivoa: prvo se uzme funkcija posla (`f`), pa sledeći
redjuser (`next`), pa tek onda element. Svaki nivo pamti prethodni u zatvorenju.

**`reduceRight` u `compose`** — spaja s desna na levo, tako da **napisani** redosled bude i redosled kroz koji
element prolazi. `compose(filter(p), map(f))` znači: prvo `filter`, pa `map`.

**Zašto baš obrnuto od običnog `compose`.** Kod običnih funkcija `compose(a, b)(x) = a(b(x))` znači „prvo `b`".
Kod transdjusera se komponuju **omotači redjusera**, pa se dvostruko obrtanje poništi i redosled ispadne
prirodan — sleva nadesno. To je poznata zamka; u proverama je posebno provereno.

### Funkcionalna paradigma prema OOP

| | Ovde (FP) | Kako bi bilo u OOP |
| --- | --- | --- |
| korak obrade | transdjuser (funkcija) | klasa koja implementira `Stage`/`Handler` |
| slaganje koraka | `compose(...)` | lanac odgovornosti, lista `Handler` objekata |
| ishod skupljanja | krajnji redjuser kao argument | posebna metoda `toList()`, `sum()` po lancu |
| broj prolaza | jedan | jedan (ali uz objekat po koraku) |

**Prednosti.** Korak obrade je jedna funkcija, bez interfejsa. Isti korak radi za bilo koji ishod i bilo koji izvor.
Nema međunizova.

**Mane.** Zapis `f => next => (acc, e) => …` je težak za čitanje dok se ne navikne, greške u tipu se vide tek
pri izvršavanju, a poruka o grešci pokazuje na anonimnu strelicu. Objektni lanac odgovornosti je razvučeniji,
ali se lakše prati u debageru.

---
# Jedinica 3 · Izravnavanje i sekvencijalno asinhrono izvršavanje

In [ ]:
const jeIterabilan = x => x != null && typeof x[Symbol.iterator] === "function";

// Workflow je iterabilan, pa se ugnježđeni workflow izravnava isto kao niz.
const izravnaj = (zadaci, dubina = 1) =>
    dubina <= 0 ? [...zadaci]
                : [...zadaci].flatMap(z => jeIterabilan(z) ? izravnaj([...z], dubina - 1) : [z]);

// Zadaci se izvršavaju REDOM: svaki sledeći dobija rezultat prethodnog.
// Akumulator preklapanja je Promise koji nosi ULAZ sledećeg koraka (uvek niz argumenata).
const izvrsiRedom = (zadaci, argumenti) =>
    [...zadaci]
        .reduce((tok, zadatak) => tok
                    .then(ulaz => zadatak.execute(...ulaz))   // sledeći korak čeka prethodni
                    .then(rezultat => [rezultat]),            // rezultat prethodnog je jedini ulaz sledećeg
                Promise.resolve(argumenti))                   // prvi zadatak dobija SVE argumente
        .then(([poslednji]) => poslednji);                    // povratna vrednost je rezultat poslednjeg

### Koncepti

**Rad nad protokolom, ne nad tipom.** `jeIterabilan` ne pita „da li je ovo Workflow" nego „da li se kroz ovo
može iterisati". Zato `izravnaj` radi i sa nizovima, i sa ugnježđenim Workflow-ovima, i sa bilo čim što neko
kasnije napravi — bez ijedne izmene.

**Asinhrono preklapanje.** Ovo je ključni mehanizam zadatka. Akumulator `reduce`-a nije vrednost nego **`Promise`**,
i svaki korak dopisuje `.then` na taj lanac. Time se sekvencijalno izvršavanje dobija bez `for` petlje i bez
uzastopnih `await`-a — kao **opis** lanca, a ne kao postupak.

**Nepromenljiv akumulator.** Kroz lanac se ne nosi rezultat nego **ulaz sledećeg koraka**, i to uvek kao **niz**
argumenata. Zbog toga prvi korak može da dobije više argumenata, a svaki sledeći tačno jedan — bez posebne
grane za prvi element.

**Rekurzija sa dubinom.** `izravnaj` prati semantiku `Array.prototype.flat`: podrazumevano jedan nivo,
sa mogućnošću dubljeg izravnavanja.

### Mehanizmi

**`reduce` sa `Promise.resolve(argumenti)` kao početnom vrednošću** — lanac počinje već „rešenim" `Promise`-om
koji nosi početne argumente.

**Dva `.then` po koraku.** Prvi izvršava zadatak, drugi pakuje rezultat u jednočlani niz da bi sledeći korak
imao isti oblik ulaza. To je mesto gde se pravilo „sledeći dobija rezultat prethodnog" i sprovodi.

**Završni `.then(([poslednji]) => poslednji)`** — razlaganje jednočlanog niza; vraća se sama vrednost, ne niz.

**`typeof x[Symbol.iterator] === "function"`** — provera protokola. `x != null` unapred, jer bi na `null`
pristup polju bacio grešku.

**Prazan spisak.** `reduce` bez ijednog koraka vraća početnu vrednost, pa `execute()` nad praznim Workflow-om
vrati prvi prosleđeni argument. Prazan Workflow se time ponaša kao neutralni element — ništa ne radi.

### Zašto ne `for … of` sa `await`

Napisano petljom bilo bi:

```js
let ulaz = argumenti;
for (const z of zadaci) ulaz = [await z.execute(...ulaz)];
return ulaz[0];
```

Radi isto, ali uvodi promenljivu koja se menja i naredbe umesto izraza. `reduce` verzija je jedan izraz i
nema stanja — cena je što se teže čita dok se obrazac „`Promise` kao akumulator" ne prepozna.

### Funkcionalna paradigma prema OOP

| | Ovde (FP) | Kako bi bilo u OOP |
| --- | --- | --- |
| redosled izvršavanja | lanac `.then` sagrađen preklapanjem | petlja sa `await` i promenljivom |
| prepoznavanje ugnježđenja | provera protokola `Symbol.iterator` | `instanceof Workflow` |
| ulaz sledećeg koraka | vrednost koja se prosleđuje | polje konteksta koje se menja (`ctx.result = …`) |

**Prednosti.** Nema promenljive koja se menja, pa nema ni pitanja „šta ako korak baci grešku na pola" —
`Promise` lanac se prekida sam. Radi sa bilo kojim iterabilnim izvorom.

**Mane.** Sekvencijalno izvršavanje znači da se nezavisni koraci **ne** izvršavaju uporedo — što je ovde traženo,
ali je i gubitak kad bi paralelizam bio moguć. Trag greške kroz lanac `.then` je manje čitljiv od
traga kroz `await` u petlji.

---
# Jedinica 4 · `Workflow` — iterabilan, nepromenljiv, sa `map`/`flat`/`flatMap`/`filter`

In [ ]:
const Workflow = (naziv, autor, zadaci = []) => {
    const z = Object.freeze([...zadaci]);            // privatna zamrznuta kopija spiska

    return Object.freeze({
        naziv, autor,

        get zadaci() { return [...z]; },             // uvek kopija ka spolja

        *[Symbol.iterator]() { yield* z; },          // iterisanjem se dobijaju pojedinačni Task objekti

        // sve četiri operacije vraćaju NOV Workflow — original ostaje netaknut
        map:     f => Workflow(naziv, autor, z.map(f)),
        filter:  p => Workflow(naziv, autor, z.filter(p)),
        flat:    (dubina = 1) => Workflow(naziv, autor, izravnaj(z, dubina)),
        flatMap: f => Workflow(naziv, autor, izravnaj(z.map(f))),

        // proizvoljna kompozicija transdjusera nad spiskom zadataka:
        //   transduce — jedan prolaz do bilo kakvog rezultata (niz, broj, tekst…)
        //   primeni   — isti prolaz, ali je rezultat opet Workflow
        transduce: (xform, krajnji = collect, pocetna = []) => z.reduce(xform(krajnji), pocetna),
        primeni:   (...transdjuseri) => Workflow(naziv, autor, z.reduce(compose(...transdjuseri)(collect), [])),

        execute: (...argumenti) => izvrsiRedom(z, argumenti)
    });
};

### Koncepti

**Iterabilni protokol.** Objekat je iterabilan ako ima metodu pod ključem `Symbol.iterator` koja vraća iterator.
Time Workflow dobija `for…of`, spread `[...wf]`, razlaganje i `Array.from` — sve besplatno, bez nasleđivanja od niza.

**Kontejner sa `map`/`filter`/`flat`/`flatMap`.** Workflow se ponaša kao kolekcija: `map` menja članove i vraća
**isti oblik** (nov Workflow), `flatMap` mapira pa izravnava. Zbog toga se operacije mogu nizati
(`wf.filter(...).map(...).flat()`), a rezultat je uvek Workflow, ne niz.

**Nepromenljivost.** Nijedna od četiri operacije ne dira original. `wf.filter(p)` je **nov** Workflow;
`wf` posle toga daje isti rezultat kao pre. To je provereno u DEO II.

**Kompozicija kao osnovna gradnja.** Workflow može da sadrži drugi Workflow, jer i Workflow ima `execute`.
`izvrsiRedom` poziva `zadatak.execute(...)` i ne pita ko je zadatak — ugnježđen Workflow prolazi kao i običan Task.
Zbog toga ugnježđena verzija i izravnata verzija daju **isti** rezultat, što je takođe provereno.

**Dva ugla na transdjusere.** `transduce` daje bilo kakav rezultat (niz naziva, broj zadataka, zbir);
`primeni` daje opet Workflow. Prvi služi za čitanje, drugi za pravljenje izmenjenog toka.

### Mehanizmi

**Generatorska metoda `*[Symbol.iterator]() { yield* z; }`** — najkraći način da se protokol ispuni.
`yield*` prosleđuje sve članove niza jedan po jedan; ne pravi se kopija i ne piše se ručni iterator sa `next()`.

**Geter `get zadaci()`** — vraća **kopiju**, pa `wf.zadaci.push(x)` ne može da dopre do originala.
Zamrzavanje objekta samo po sebi ne bi zaštitilo niz iza polja.

**Rekurzivni poziv `Workflow(...)`** u sve četiri operacije — `naziv` i `autor` se prenose, pa izvedeni tok
zadržava identitet svog izvora.

**`primeni` sa podrazumevanim `collect`** — bez argumenata `compose()` vraća identitet, pa `primeni()`
daje kopiju spiska. Nema posebne grane za taj slučaj.

**`flat(dubina = 1)`** — podrazumevana vrednost prati `Array.prototype.flat`.

### Pretpostavka

Tekst traži „operacije map, flat, flatMap i filter", ali ne kaže šta vraćaju. Uzeto je da vraćaju **Workflow**,
a ne niz — tako se mogu nizati i tako `flat` ima smisla nad ugnježđenim Workflow-ovima. Za pristup golom spisku
tu je geter `zadaci` i iterisanje.

### Funkcionalna paradigma prema OOP

| | Ovde (FP) | Kako bi bilo u OOP |
| --- | --- | --- |
| iterabilnost | `Symbol.iterator` na običnom objektu | nasleđivanje `Iterable`/`Collection` |
| izmena spiska | nov Workflow | `wf.addTask(t)` nad istom instancom |
| ugnježđenje | Workflow je i sam „zadatak" jer ima `execute` | zajednički interfejs `Izvrsivo` koji oba implementiraju |
| proširenje ponašanja | transdjuser spolja | podklasa `LoggingWorkflow extends Workflow` |
| privatnost spiska | zatvorenje + geter sa kopijom | `private readonly List` + `Collections.unmodifiableList` |

**Prednosti.** Ugnježđenje radi bez ijednog interfejsa — dovoljno je da objekat ima `execute`. Nema deljene
promenljive liste zadataka. Ponašanje se dodaje spolja, pa nema kombinatorne eksplozije podklasa
(`LoggingWorkflow`, `TimedWorkflow`, `LoggingTimedWorkflow`…).

**Mane.** Svaka operacija kopira spisak. Nema provere da su članovi zaista `Task` — greška izlazi pri izvršavanju.
Objektni pristup bi dao jasniji tip i pomoć editora.

---
# Jedinica 5 · Transdjuser za ispis svakog koraka

In [ ]:
// Traženi transdjuser: svaki Task zamenjuje NOVIM Task-om koji oko
// izvršavanja dopisuje ispis ulaza i izlaza. Original se ne dira.
const saIspisom = map(zadatak => Task(zadatak.naziv, async (...argumenti) => {
    console.log(`   → ${zadatak.naziv} ulaz:`, ...argumenti);
    const rezultat = await zadatak.execute(...argumenti);
    console.log(`   ← ${zadatak.naziv} izlaz:`, rezultat);
    return rezultat;
}));

// isti obrazac, drugi posao — dokaz da mehanizam prima proizvoljan transdjuser
const merenje = map(zadatak => Task(zadatak.naziv, async (...argumenti) => {
    const pocetak = Date.now();
    const rezultat = await zadatak.execute(...argumenti);
    console.log(`   ⏱ ${zadatak.naziv}: ${Date.now() - pocetak} ms`);
    return rezultat;
}));

### Koncepti

**Dekorisanje bez izmene.** Novi Task ima isti naziv i isti ugovor, a unutra poziva stari.
Original ostaje netaknut i dalje upotrebljiv — nije „ugašen" ni zamenjen, samo obmotan u novoj verziji toka.

**Presečna briga (cross-cutting concern) izdvojena na jedno mesto.** Ispis se tiče svakog koraka, a nije posao
nijednog. Da je pisan u samim definicijama, bio bi razasut i ne bi se mogao isključiti. Ovako se dodaje i sklanja
jednim argumentom.

**Transdjuser kao mesto proširenja.** `saIspisom` **jeste** `map(...)` — obična upotreba već postojećeg transdjusera.
Nije bilo potrebe ništa dograditi u `Workflow`-u: mesto proširenja je već postojalo.

**Zadržan ugovor.** Obmotani zadatak vraća **isti rezultat** kao original, pa se ceo tok ponaša isto —
razlika je samo u ispisu. To je i provereno: workflow sa ispisom daje isti rezultat kao bez njega.

### Mehanizmi

**`map(f)` gde `f` prima Task i vraća Task** — transdjuser radi nad spiskom zadataka, ne nad podacima koje
zadaci obrađuju. To je razlika koju je lako promašiti: element toka ovde je **zadatak**.

**`async` + `await zadatak.execute(...)`** — obmotač mora da sačeka original da bi mogao da ispiše izlaz.

**Zatvorenje nad `zadatak`** — nova definicija pamti stari zadatak; original se nigde ne menja.

**Spread u `console.log(..., ...argumenti)`** — argumenti se ispisuju kao zasebne vrednosti, pa se u konzoli
vide kao objekti koji se mogu raširiti, a ne kao tekst.

### Zašto ovo dokazuje da kompozicija radi

`merenje` je napisan po istom obrascu, ali radi nešto drugo. Pošto su oba obični transdjuseri, mogu se
kombinovati sa `filter`-om i međusobno, u bilo kom redosledu — što je u DEO II i pokazano
(`primeni(filter(...), saIspisom)`). Da je ispis bio ugrađen u `Workflow.execute`, ništa od toga ne bi bilo moguće.

### Funkcionalna paradigma prema OOP

| | Ovde (FP) | Kako bi bilo u OOP |
| --- | --- | --- |
| dodavanje ispisa | transdjuser koji vraća nov Task | `LoggingTaskDecorator implements Task` |
| kombinovanje sa drugim ponašanjem | `compose(a, b)` | ugnježđeni dekoratori ili AOP savet |
| uključivanje/isključivanje | argument u `primeni` | izgradnja drugog lanca objekata |

**Prednosti.** Dekorator je jedan izraz, bez klase i interfejsa. Kombinuje se sa bilo kojim drugim transdjuserom.
Original ostaje čist i proverljiv.

**Mane.** Obmotani zadatak nije `===` originalu, pa poređenje po referenci prestaje da važi — u DEO II je zato
posebno provereno da originalni spisak **nije** obmotan. Ispis unutar transdjusera je efekat, pa taj transdjuser
nije čist.

---
# DEO II — PROVERE

Provera je poređenje **dobijenog** i **očekivanog**, sa izveštajem `N/N provera prošlo` na kraju.

**Asinhrono testiranje.** Svaka provera koja dodiruje `execute` mora da `await`-uje rezultat — inače bi se
poredio `Promise`, a ne vrednost. U HTML fajlu je zbog toga sve u funkciji `provere()`; ovde kernel dozvoljava
`await` na najvišem nivou, pa ćelije stoje direktno.

In [ ]:
// ── alat za provere ───────────────────────────────────────────────────
let ukupno = 0, prosle = 0;

const isti = (a, b) => JSON.stringify(a) === JSON.stringify(b);

const proveri = (opis, dobijeno, ocekivano) => {
    ukupno++;
    const ok = isti(dobijeno, ocekivano);
    if (ok) prosle++;
    console.log(`${ok ? "✔" : "✘"} ${opis}\n     dobijeno:  ${JSON.stringify(dobijeno)}` +
                (ok ? "" : `\n     očekivano: ${JSON.stringify(ocekivano)}`));
};

const naslov = tekst => console.log(`\n═══ ${tekst} ═══`);

In [ ]:
// ── nekoliko Task objekata ────────────────────────────────────────────
const parsiraj   = Task("parsiraj",   tekst => tekst.split(",").map(Number));
const bezNula    = Task("bezNula",    niz => niz.filter(x => x !== 0));
const kvadrati   = Task("kvadrati",   niz => niz.map(x => x * x));
const zbir       = Task("zbir",       niz => niz.reduce((s, x) => s + x, 0));
const formatiraj = Task("formatiraj", x => `ukupno: ${x}`);
const cekaj      = Task("cekaj",      async x => { await new Promise(k => setTimeout(k, 20)); return x; });
const spoji      = Task("spoji",      (a, b) => `${a}-${b}`);
const saberi     = Task("saberi",     (a, b, c) => a + b + c);
const puta10     = Task("puta10",     x => x * 10);

// ── nekoliko Workflow objekata ────────────────────────────────────────
const priprema = Workflow("priprema", "Blagoje", [parsiraj, bezNula]);
const racun    = Workflow("racun",    "Blagoje", [kvadrati, cekaj, zbir]);
const glavni   = Workflow("glavni",   "Blagoje", [priprema, racun, formatiraj]);   // ugnježđen
const ravan    = glavni.flat();                                                    // izravnat

console.log("ugnježđen:", [...glavni].map(z => z.naziv));
console.log("izravnat: ", [...ravan].map(z => z.naziv));

### Kako su podaci sastavljeni

Zadaci su namerno **sitni i čisti** — svaki radi jednu stvar nad vrednošću koju dobije.
Tok `parsiraj → bezNula → kvadrati → cekaj → zbir → formatiraj` obrađuje `"1,0,2,3"`:

| korak | ulaz | izlaz |
| --- | --- | --- |
| `parsiraj` | `"1,0,2,3"` | `[1, 0, 2, 3]` |
| `bezNula` | `[1, 0, 2, 3]` | `[1, 2, 3]` |
| `kvadrati` | `[1, 2, 3]` | `[1, 4, 9]` |
| `cekaj` | `[1, 4, 9]` | `[1, 4, 9]` (posle 20 ms) |
| `zbir` | `[1, 4, 9]` | `14` |
| `formatiraj` | `14` | `"ukupno: 14"` |

`glavni` je **ugnježđen**: sadrži dva Workflow-a i jedan Task. `ravan` je isti tok posle `flat()`.
Oba daju isti rezultat — jedno preko ugnježđenog `execute`-a, drugo preko izravnatog spiska.

In [ ]:
// ── 1 · Task ──────────────────────────────────────────────────────────
naslov("1 · Task");

proveri("naziv i definicija su dostupni",
        [parsiraj.naziv, typeof parsiraj.definicija], ["parsiraj", "function"]);

proveri("rezultat execute je rezultat poziva definicije",
        await kvadrati.execute([2, 3]), kvadrati.definicija([2, 3]));

proveri("execute prima jedan argument", await parsiraj.execute("1,0,2"), [1, 0, 2]);

proveri("execute prima proizvoljan broj argumenata",
        [await spoji.execute("A", "B"), await saberi.execute(1, 2, 3)], ["A-B", 6]);

proveri("execute je asinhron — vraća Promise", parsiraj.execute("1") instanceof Promise, true);

proveri("definicija sme i sama biti asinhrona", await cekaj.execute(7), 7);

proveri("Task je zamrznut", Object.isFrozen(parsiraj), true);

// ── 2 · Workflow: podaci i iterabilnost ───────────────────────────────
naslov("2 · Workflow i iterabilnost");

for (const z of priprema) console.log("   član:", z.naziv);

proveri("naziv i autor", [priprema.naziv, priprema.autor], ["priprema", "Blagoje"]);

proveri("spisak Task objekata", priprema.zadaci.map(z => z.naziv), ["parsiraj", "bezNula"]);

proveri("iterisanjem se dobijaju pojedinačni Task objekti",
        [...priprema].map(z => z.naziv), ["parsiraj", "bezNula"]);

proveri("iterisani članovi su pravi Task objekti",
        [...priprema].every(z => typeof z.definicija === "function" && typeof z.execute === "function"), true);

priprema.zadaci.push(formatiraj);                    // NAMERNO: geter vraća kopiju
proveri("geter zadaci vraća kopiju — workflow ostaje isti", [...priprema].length, 2);

proveri("Workflow je zamrznut", Object.isFrozen(priprema), true);

In [ ]:
// ── 3 · map, filter, flat, flatMap ────────────────────────────────────
naslov("3 · map, filter, flat, flatMap");

proveri("flat izravnava ugnježđene workflow objekte",
        [...ravan].map(z => z.naziv),
        ["parsiraj", "bezNula", "kvadrati", "cekaj", "zbir", "formatiraj"]);

proveri("filter izdvaja zadatke",
        [...ravan.filter(z => z.naziv !== "cekaj")].map(z => z.naziv),
        ["parsiraj", "bezNula", "kvadrati", "zbir", "formatiraj"]);

proveri("map preslikava zadatke",
        [...ravan.map(z => Task(z.naziv.toUpperCase(), z.definicija))].map(z => z.naziv),
        ["PARSIRAJ", "BEZNULA", "KVADRATI", "CEKAJ", "ZBIR", "FORMATIRAJ"]);

proveri("flatMap preslikava pa izravnava",
        [...priprema.flatMap(z => Workflow("par", "Blagoje", [z, Task("kroz", x => x)]))].map(z => z.naziv),
        ["parsiraj", "kroz", "bezNula", "kroz"]);

proveri("sve četiri vraćaju NOV Workflow, sa istim nazivom i autorom",
        [ravan.naziv, ravan.autor, ravan !== glavni], ["glavni", "Blagoje", true]);

proveri("original je netaknut", [...glavni].map(z => z.naziv), ["priprema", "racun", "formatiraj"]);

In [ ]:
// ── 4 · execute ───────────────────────────────────────────────────────
naslov("4 · execute");

proveri("lančano izvršavanje: izlaz jednog je ulaz sledećeg",
        await ravan.execute("1,0,2,3"), "ukupno: 14");   // [1,0,2,3]→[1,2,3]→[1,4,9]→14→tekst

proveri("ugnježđen workflow se izvršava isto kao izravnat",
        await glavni.execute("1,0,2,3"), await ravan.execute("1,0,2,3"));

proveri("prvi zadatak dobija SVE prosleđene argumente",
        await Workflow("tri ulaza", "Blagoje", [saberi, puta10]).execute(2, 3, 5), 100);

proveri("povratna vrednost je rezultat poslednjeg zadatka",
        await Workflow("dva koraka", "Blagoje", [parsiraj, zbir]).execute("1,2,3"), 6);

proveri("execute je asinhron — vraća Promise", ravan.execute("1") instanceof Promise, true);

proveri("izmenjen spisak daje drugačiji rezultat, original isti",
        [await ravan.filter(z => z.naziv !== "cekaj").execute("4,0,5"), await ravan.execute("4,0,5")],
        ["ukupno: 41", "ukupno: 41"]);

// dokaz da se zadaci izvršavaju REDOM, a ne uporedo:
// spor zadatak je prvi i njegov trag mora stići pre traga brzog
const trag = [];
const spor = Task("spor", async x => { await new Promise(k => setTimeout(k, 30)); trag.push("spor"); return x; });
const brz  = Task("brz",  x => (trag.push("brz"), x));
await Workflow("redosled", "Blagoje", [spor, brz]).execute(1);

proveri("sledeći zadatak čeka prethodni da se završi", trag, ["spor", "brz"]);

### Šta ove provere zapravo dokazuju

**„ugnježđen workflow se izvršava isto kao izravnat"** — jedina provera koja pokazuje da je Workflow
zamenljiv za Task u lancu. Radi zato što `izvrsiRedom` traži samo `execute`, a ne tip.

**„prvi zadatak dobija SVE prosleđene argumente"** — `saberi` prima tri argumenta, `puta10` jedan.
Da je akumulator nosio golu vrednost umesto niza argumenata, ova provera bi pukla.

**„sledeći zadatak čeka prethodni"** — po rezultatu se sekvencijalnost ne vidi, jer bi i uporedno izvršavanje
dalo istu vrednost. Vidi se po **redosledu tragova**: spor korak je prvi, i njegov trag ipak stiže prvi.
Da je `execute` bio pisan sa `Promise.all`, ovde bi izašlo `["brz", "spor"]`.

In [ ]:
// ── 5 · proizvoljna kompozicija transdjusera ──────────────────────────
naslov("5 · transdjuseri");

proveri("transduce sa jednim transdjuserom — nazivi zadataka",
        ravan.transduce(map(z => z.naziv)),
        ["parsiraj", "bezNula", "kvadrati", "cekaj", "zbir", "formatiraj"]);

proveri("transduce sa drugim krajnjim redjuserom — broj zadataka",
        ravan.transduce(map(() => 1), sum, 0), 6);

proveri("kompozicija filter → map u jednom prolazu",
        ravan.transduce(compose(filter(z => z.naziv !== "cekaj"), map(z => z.naziv))),
        ["parsiraj", "bezNula", "kvadrati", "zbir", "formatiraj"]);

proveri("primeni bez transdjusera vraća isti spisak", ravan.primeni().zadaci.length, 6);

console.log("primeni(saIspisom) — ispis svakog koraka izvršavanja:");
const saIspisomWf = ravan.primeni(saIspisom);
proveri("workflow sa ispisom daje isti rezultat", await saIspisomWf.execute("1,0,2,3"), "ukupno: 14");

console.log("primeni(filter, saIspisom) — proizvoljna kompozicija:");
const bezCekanja = ravan.primeni(filter(z => z.naziv !== "cekaj"), saIspisom);
proveri("kompozicija transdjusera menja i spisak i ponašanje",
        [bezCekanja.zadaci.length, await bezCekanja.execute("4,0,5")], [5, "ukupno: 41"]);

console.log("primeni(merenje) — trajanje svakog koraka:");
proveri("drugi transdjuser, isti mehanizam", await ravan.primeni(merenje).execute("1,0,2,3"), "ukupno: 14");

In [ ]:
// isti obrazac kao saIspisom, ali beleži u niz — da se ispis može i PROVERITI
const dnevnik = [];
const saBelezenjem = map(zadatak => Task(zadatak.naziv, async (...argumenti) => {
    dnevnik.push(`→ ${zadatak.naziv}`);
    const rezultat = await zadatak.execute(...argumenti);
    dnevnik.push(`← ${zadatak.naziv}`);
    return rezultat;
}));

await ravan.primeni(saBelezenjem).execute("1,0,2,3");

proveri("transdjuser je obuhvatio svaki korak izvršavanja",
        dnevnik,
        ["→ parsiraj", "← parsiraj", "→ bezNula", "← bezNula", "→ kvadrati", "← kvadrati",
         "→ cekaj", "← cekaj", "→ zbir", "← zbir", "→ formatiraj", "← formatiraj"]);

// ── 6 · imutabilnost ──────────────────────────────────────────────────
naslov("6 · imutabilnost");

proveri("posle svih operacija original je nepromenjen",
        [...ravan].map(z => z.naziv),
        ["parsiraj", "bezNula", "kvadrati", "cekaj", "zbir", "formatiraj"]);

proveri("ugnježđen workflow nepromenjen", [...glavni].map(z => z.naziv), ["priprema", "racun", "formatiraj"]);

proveri("originalni zadaci nisu obmotani", ravan.zadaci[0] === parsiraj, true);

proveri("original i dalje daje isti rezultat", await ravan.execute("1,0,2,3"), "ukupno: 14");

// ── izveštaj ──────────────────────────────────────────────────────────
console.log(`\n═══ ${prosle}/${ukupno} provera prošlo ═══` +
            (prosle === ukupno ? " sve ispravno" : " IMA GREŠAKA"));

### Zašto `saBelezenjem` pored `saIspisom`

`saIspisom` ispisuje u konzolu — to se vidi, ali se ne može uporediti sa očekivanim.
`saBelezenjem` je **isti obrazac** koji umesto ispisa upisuje u niz, pa provera može da tvrdi da je transdjuser
obuhvatio **tačno dvanaest** koraka, u tačnom redosledu. Time je zahtev „transdjuser za ispis svakog koraka"
i pokazan i dokazan.

Usput to pokazuje i glavnu prednost transdjusera: promena cilja (konzola → niz) nije tražila nijednu izmenu
ni u `Workflow`-u ni u zadacima.

### „originalni zadaci nisu obmotani"

`ravan.zadaci[0] === parsiraj` je `true` i posle svih `primeni` poziva. Obmotavanje je napravilo **nove**
zadatke u **novom** Workflow-u; original je ostao netaknut. Da je `primeni` menjao spisak na mestu,
ova provera bi bila jedina koja bi to otkrila.

---
# Rečnik pojmova iz ovog zadatka

| Pojam | Značenje | Gde se vidi |
| --- | --- | --- |
| **prvorazredna funkcija** | funkcija je vrednost kao i svaka druga | `definicija` kao polje `Task`-a |
| **funkcija višeg reda** | prima ili vraća funkciju | `map`, `filter`, `compose`, `primeni`, `transduce` |
| **redjuser** | `(acc, e) => acc` — jedini oblik koji `reduce` razume | `collect`, `sum` |
| **transdjuser** | funkcija koja prima redjuser i vraća redjuser | `map(f)`, `filter(p)`, `saIspisom`, `merenje` |
| **kompozicija** | spajanje funkcija u novu funkciju | `compose(...)`, `primeni(a, b)` |
| **iterabilni protokol** | objekat sa `Symbol.iterator` radi u `for…of` i spread-u | `*[Symbol.iterator]()` u `Workflow` |
| **generator** | funkcija koja daje vrednosti jednu po jednu (`yield`) | isti taj `Symbol.iterator` |
| **asinhrono preklapanje** | `reduce` čiji je akumulator `Promise` | `izvrsiRedom` |
| **nepromenljivost** | operacija vraća novu vrednost umesto izmene | `map`/`filter`/`flat`/`flatMap`/`primeni` |
| **dekorisanje** | ponašanje se dodaje obmotavanjem, bez izmene originala | `saIspisom`, `merenje`, `saBelezenjem` |
| **rest / spread** | proizvoljan broj argumenata unutra i napolje | `execute: async (...argumenti) => definicija(...argumenti)` |

# Šta zadatak zapravo proverava

1. **Da li se posao može opisati funkcijom umesto podklasom** — `definicija` kao podatak (jedinica 1).
2. **Da li se sekvenca može opisati bez petlje** — `reduce` nad lancem `Promise`-a (jedinica 3).
3. **Da li se protokol razume** — `Symbol.iterator` čini da `for…of`, spread i `flat` rade sami (jedinica 4).
4. **Da li se ponašanje dodaje spolja** — transdjuser umesto `if (logovanje)` u `execute` (jedinica 5).
5. **Da li se „proizvoljna kompozicija" shvata doslovno** — `primeni(...transdjuseri)` bez ograničenja broja.

# Veza sa Kolokvijumom I

Mehanizmi odavde rešavaju dva mesta iz prvog zadatka:

- **`pretraga` sa `n` kriterijuma** pravi `n` međunizova. Ista pretraga preko transdjusera
  (`compose(filter(k1), filter(k2))(collect)`) prolazi **jednom**, bez ijednog međuniza.
- **`stanjeOd`** je već redjuser; sa transdjuserima bi se filtriranje realizovanih i sabiranje spojili
  u isti prolaz, bez izdvojenog `filter` koraka.

Isto važi i obrnuto: **nepromenljivost** i **fabričke funkcije** iz K1 su ovde iskorišćene bez izmene —
`Workflow` je ista vrsta objekta kao `GlavnaKnjiga`, samo sa drugim sadržajem.